<a href="https://colab.research.google.com/github/Alokarya123-nitw/Subscription-SaaS-Product-Led-Growth-PLG-Seat-Expansion-Predictor/blob/main/SaaS_PLG_Pipeline_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
n_records = 10000

# 1. Generate core workspace fields
workspace_ids = np.arange(1001, 1001 + n_records)

# Mocking messy corporate and personal email domains
domains = ['@gmail.com', '@yahoo.com', '@microsoft.com', '@uber.com',
           '@meta.com', '@accenture.com', '@netflix.com', '@unknown_typo', '']
workspace_domains = np.random.choice(domains, size=n_records, p=[0.40, 0.15, 0.10, 0.08, 0.07, 0.08, 0.07, 0.03, 0.02])

# Active users on the free tier (some zero or empty, simulating anomalies)
active_users = np.random.negative_binomial(n=5, p=0.3, size=n_records)
active_users = np.where(workspace_domains == '@gmail.com', active_users // 2, active_users) # personal domains have fewer users

# Storage used in GB (some missing values introduced as NaNs)
storage_used = np.random.exponential(scale=15, size=n_records)
storage_used[np.random.choice(n_records, size=500, replace=False)] = np.nan

# Telemetry events over the last 30 days
weekly_invites = np.random.poisson(lam=3, size=n_records)
weekly_invites = np.where(active_users > 10, weekly_invites * 3, weekly_invites)
advanced_feature_clicks = np.random.poisson(lam=5, size=n_records)

# Country tracking with casing errors
countries = ['INDIA', 'India', 'USA', 'usa', 'united states', 'UK', 'uk', 'GERMANY', None]
workspace_countries = np.random.choice(countries, size=n_records)

# Compile into a DataFrame
df = pd.DataFrame({
    'workspace_id': workspace_ids,
    'company_domain': workspace_domains,
    'active_users': active_users,
    'storage_used_gb': np.round(storage_used, 2),
    'weekly_invites_sent': weekly_invites,
    'advanced_clicks': advanced_feature_clicks,
    'country': workspace_countries
})

# Save to a raw CSV file in the Colab cloud directory
df.to_csv('raw_saas_telemetry.csv', index=False)
print("SUCCESS: 'raw_saas_telemetry.csv' generated with 10,000 rows. Download it from the left panel sidebar.")


SUCCESS: 'raw_saas_telemetry.csv' generated with 10,000 rows. Download it from the left panel sidebar.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. Load the clean filtered data
df_model = pd.read_csv('safe_saas_telemetry.csv')

# 2. Simulate the real-world conversion targets (around 3.5% baseline upgrades)
np.random.seed(42)
conversion_score = (df_model['active_users'] * 0.4) + (df_model['weekly_invites_sent'] * 0.5) + (df_model['advanced_clicks'] * 0.3)
conversion_threshold = np.percentile(conversion_score, 96.5)
df_model['converted_enterprise'] = (conversion_score >= conversion_threshold).astype(int)

# 3. Define features and target variable
X = df_model[['active_users', 'storage_used_gb', 'weekly_invites_sent', 'advanced_clicks']].fillna(0)
y = df_model['converted_enterprise']

# Split data into train and test evaluation splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale continuous input vectors
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train model with strict class-weight compensation parameters
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate classification outcomes
y_pred = model.predict(X_test_scaled)
print("=== PRODUCTION CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

# 5. Extract continuous upgrade probability metrics (The Propensity Score)
X_all_scaled = scaler.transform(X)
df_model['upgrade_probability'] = np.round(model.predict_proba(X_all_scaled)[:, 1], 4)

# Save the final file containing prediction scores
df_model.to_csv('predicted_saas_conversions.csv', index=False)
print("\nSUCCESS: 'predicted_saas_conversions.csv' generated with Propensity Scores. Refresh and check the left panel.")


=== PRODUCTION CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       814
           1       0.67      1.00      0.80        30

    accuracy                           0.98       844
   macro avg       0.83      0.99      0.90       844
weighted avg       0.99      0.98      0.98       844


SUCCESS: 'predicted_saas_conversions.csv' generated with Propensity Scores. Refresh and check the left panel.
